# Feature Engineering and Aggregation

This notebook performs feature engineering on the preprocessed taxi-weather dataset and 
aggregates the data for subsequent analysis and modelling steps.

The preprocessed data is split into training and test sets, missing values in weather features 
are imputed, a weather severity score is created, trips are aggregated and earnings potential is engineered.

The resulting training and test datasets are saved as Parquet files.

In [1]:
# Importing the libraries required throughout the notebook
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialising the Spark session for feature engineering and aggregation
spark = (
    SparkSession.builder
    .appName("MAST30034_Project1_FeatureEngineering")
    .config("spark.driver.memory", "1500m")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/08/31 18:17:38 WARN Utils: Your hostname, MAHIKA-F8R8E40, resolves to a loopback address: 127.0.1.1; using 192.168.21.80 instead (on interface eth0)
26/08/31 18:17:38 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/08/31 18:17:44 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/08/31 18:17:46 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/08/31 18:17:46 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


In [2]:
# Loading the preprocessed taxi-weather dataframe

taxi_weather = spark.read.parquet("../data/processed/taxi_weather.parquet")

print(f"Rows:    {taxi_weather.count():,}")
print(f"Columns: {len(taxi_weather.columns)}")

taxi_weather.printSchema()

Rows:    16,772,341
Columns: 17
root
 |-- pickup_date: date (nullable = true)
 |-- pickup_hour: integer (nullable = true)
 |-- time_of_day: string (nullable = true)
 |-- week_day: integer (nullable = true)
 |-- trip_duration_min: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- trip_speed_mph: double (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- NAME: string (nullable = true)
 |-- LATITUDE: double (nullable = true)
 |-- LONGITUDE: double (nullable = true)
 |-- weather_datetime: timestamp (nullable = true)
 |-- HourlyPrecipitation: double (nullable = true)
 |-- temp_celsius: double (nullable = true)
 |-- HourlyWindSpeed: double (nullable = true)
 |-- precip_type: string (nullable = true)



## 1. Train and Test Split

In [3]:
# Adding a timestamp column to determine the date for an 80-20 split
taxi_weather_with_timestamp = taxi_weather.withColumn(
    "pickup_timestamp",
    F.unix_timestamp(F.col("pickup_date").cast("timestamp"))
)

date_quantiles = taxi_weather_with_timestamp.approxQuantile("pickup_timestamp", [0.8], 0.001)

split_timestamp = date_quantiles[0]

# Converting the 80th percentile timestamp back to a date
split_date = spark.sql(f"SELECT to_date(from_unixtime({split_timestamp})) AS split_date"
                       ).collect()[0]["split_date"]

print(f"Split date (80th percentile): {split_date}")

# Splitting the data into training and test sets
train_df = taxi_weather.filter(F.col("pickup_date") < F.lit(split_date))
test_df = taxi_weather.filter(F.col("pickup_date") >= F.lit(split_date))

train_count = train_df.count()
test_count = test_df.count()
total_count = taxi_weather.count()

print(f"\nTraining rows:  {train_count:,} ({train_count/total_count*100:.2f}%)")
print(f"Testing rows:   {test_count:,} ({test_count/total_count*100:.2f}%)")
print(f"Total:          {train_count + test_count:,}")

print("\nTrain date range:")
train_df.select(
    F.min("pickup_date").alias("earliest_date"),
    F.max("pickup_date").alias("latest_date")
).show()

print("Test date range:")
test_df.select(
    F.min("pickup_date").alias("earliest_date"),
    F.max("pickup_date").alias("latest_date")
).show()

Split date (80th percentile): 2024-03-26



Training rows:  13,397,373 (79.88%)
Testing rows:   3,374,968 (20.12%)
Total:          16,772,341

Train date range:


+-------------+-----------+
|earliest_date|latest_date|
+-------------+-----------+
|   2023-11-01| 2024-03-25|
+-------------+-----------+

Test date range:


+-------------+-----------+
|earliest_date|latest_date|
+-------------+-----------+
|   2024-03-26| 2024-04-30|
+-------------+-----------+



## 2. Weather Feature Imputation

In [4]:
# Calculating the medians for missing weather values from the training set
weather_medians = train_df.select(
    F.percentile_approx("HourlyWindSpeed", 0.5).alias("wind_median"),
    F.percentile_approx("HourlyPrecipitation", 0.5).alias("precip_median"),
).collect()[0]

wind_median = weather_medians["wind_median"]
precip_median = weather_medians["precip_median"]

print(f"Wind median:          {wind_median:.2f}")
print(f"Precipitation median: {precip_median:.2f}")

print(
    "Training set values missing before imputation:",
    train_df.filter(
        F.col("HourlyWindSpeed").isNull() | F.col("HourlyPrecipitation").isNull() 
    ).count()
)

print(
    "Test set values missing before imputation:",
    test_df.filter(
        F.col("HourlyWindSpeed").isNull() | F.col("HourlyPrecipitation").isNull() 
    ).count()
)

# Imputing missing weather values using the computed medians
train_df = train_df.fillna({
    "HourlyWindSpeed": wind_median, 
    "HourlyPrecipitation": precip_median
    })

test_df = test_df.fillna({
    "HourlyWindSpeed": wind_median, 
    "HourlyPrecipitation": precip_median
    })

print(
    "Training set values missing after imputation:",
    train_df.filter(
        F.col("HourlyWindSpeed").isNull() | F.col("HourlyPrecipitation").isNull()
    ).count()
)

print(
    "Test set values missing after imputation:",
    test_df.filter(
        F.col("HourlyWindSpeed").isNull() | F.col("HourlyPrecipitation").isNull()
    ).count()
)

Wind median:          2.60
Precipitation median: 0.00


Training set values missing before imputation: 1243761


Test set values missing before imputation: 393405
Training set values missing after imputation: 0
Test set values missing after imputation: 0


## 3. Severity Score

In [5]:
# Adding temperature severity column

def add_temp_severity(df):

    """
    This function calculates temperature severity based on how far the temperature is
    outside the pleasant 15 to 25 degree Celsius range.

    Parameters:
    df: Current PySpark DataFrame

    Returns: 
    PySpark DataFrame with the temp_severity feature added
    
    """ 

    return df.withColumn(
        "temp_severity",
        F.when(F.col("temp_celsius") < 15, 15 - F.col("temp_celsius"))
         .when(F.col("temp_celsius") > 25, F.col("temp_celsius") - 25)
         .otherwise(F.lit(0.0))
    )

train_df = add_temp_severity(train_df)
test_df = add_temp_severity(test_df)

In [6]:
# Calculating the training set ranges for min-max scaling the continuous weather features
normalization_range = train_df.select(
    F.min("temp_severity").alias("temp_min_sev"),
    F.max("temp_severity").alias("temp_max_sev"),
    F.min("HourlyPrecipitation").alias("precip_min"),
    F.max("HourlyPrecipitation").alias("precip_max"),
    F.min("HourlyWindSpeed").alias("wind_min_spd"),
    F.max("HourlyWindSpeed").alias("wind_max_spd"),
).collect()[0]

temp_min_sev, temp_max_sev = normalization_range["temp_min_sev"], normalization_range["temp_max_sev"]
precip_min, precip_max = normalization_range["precip_min"], normalization_range["precip_max"]
wind_min_spd, wind_max_spd = normalization_range["wind_min_spd"], normalization_range["wind_max_spd"]

def apply_min_max_scaling(df):

    """
    This function applies min-max scaling to the continuous weather features using
    the ranges calculated from the training set.

    df: Current PySpark DataFrame

    Returns: 
    PySpark DataFrame with the normalised weather features added
    
    """

    return (
        df
        .withColumn("temp_norm", (F.col("temp_severity") - temp_min_sev) / (temp_max_sev - temp_min_sev))
        .withColumn("precip_norm", (F.col("HourlyPrecipitation") - precip_min) / (precip_max - precip_min))
        .withColumn("wind_norm", (F.col("HourlyWindSpeed") - wind_min_spd) / (wind_max_spd - wind_min_spd))
    )

train_df = apply_min_max_scaling(train_df)
test_df = apply_min_max_scaling(test_df)



In [7]:

def add_severity_score(df):

    """
    This function calculates a weather severity score by averaging the normalised
    temperature, precipitation and wind features.

    Parameters:
    df: Current PySpark DataFrame

    Returns:
    PySpark DataFrame with the severity_score feature added
    
    """

    return df.withColumn(
        "severity_score",
        (F.col("temp_norm") + F.col("precip_norm") + F.col("wind_norm")) / 3
    )

train_df = add_severity_score(train_df)
test_df = add_severity_score(test_df)

print("Training set severity_score summary:")
train_df.select("severity_score").summary("min", "25%", "50%", "75%", "90%", "99.9%", "max").show()

print("Test set severity_score summary:")
test_df.select("severity_score").summary("min", "25%", "50%", "75%", "90%", "99.9%", "max").show()

# Calculating percentiles from the training set for categorising severity_score
severity_cutoffs = train_df.select(
    F.percentile_approx("severity_score", 1/4).alias("q25"),   
    F.percentile_approx("severity_score", 1/2).alias("q50"),   
    F.percentile_approx("severity_score", 3/4).alias("q75"),   
).collect()[0]

q25, q50, q75 = severity_cutoffs["q25"], severity_cutoffs["q50"], severity_cutoffs["q75"]

def add_severity_cat(df):

    """
    This function categorises the continuous severity score into four categories based
    on the percentiles calculated from the training set.

    Parameters:
    df: Current PySpark DataFrame

    Returns:
    PySpark DataFrame with the severity_cat feature added
    
    """


    return df.withColumn(
        "severity_cat",
        F.when(F.col("severity_score") <= q25, "normal")
         .when(F.col("severity_score") <= q50, "mild")
         .when(F.col("severity_score") <= q75, "moderate")
         .otherwise("high")
    )

train_df = add_severity_cat(train_df)
test_df = add_severity_cat(test_df)

print("Training set severity category distribution:")
train_df.groupBy("severity_cat").count().orderBy("severity_cat").show()

print("Test set severity category distribution:")
test_df.groupBy("severity_cat").count().orderBy("severity_cat").show()

Training set severity_score summary:


+-------+-------------------+
|summary|     severity_score|
+-------+-------------------+
|    min|                0.0|
|    25%|0.11695655447631954|
|    50%|0.17131993072810783|
|    75%|0.22176793916120774|
|    90%| 0.2709735712672238|
|  99.9%| 0.4367592801746856|
|    max| 0.5174740047292627|
+-------+-------------------+

Test set severity_score summary:


+-------+--------------------+
|summary|      severity_score|
+-------+--------------------+
|    min|                 0.0|
|    25%|0.060682810380939735|
|    50%| 0.09625028235825613|
|    75%| 0.14843008809577593|
|    90%| 0.18927582688120914|
|  99.9%|  0.3477922147932581|
|    max| 0.46165374190975844|
+-------+--------------------+



Training set severity category distribution:


+------------+-------+
|severity_cat|  count|
+------------+-------+
|        high|3304034|
|        mild|3369741|
|    moderate|3367245|
|      normal|3356353|
+------------+-------+

Test set severity category distribution:


+------------+-------+
|severity_cat|  count|
+------------+-------+
|        high| 165788|
|        mild| 844958|
|    moderate| 354227|
|      normal|2009995|
+------------+-------+



## 4. Trip Aggregation and Earnings Potential

In [8]:
# Aggregating training taxi trips by location, severity, day, precipitation type and time of day
train_agg = (
    train_df
    .groupBy(
        "PULocationID", "severity_cat", "week_day",
        "precip_type", "time_of_day"
    )
    .agg(
        F.sum("fare_amount").alias("total_fare"),
        F.sum("trip_duration_min").alias("total_duration_min"),
        F.count("*").alias("trip_count"),
        F.avg("severity_score").alias("avg_severity_score")
    )
    # Calculating earnings potential as fare earned per active hour
    .withColumn(
        "earnings_potential",              
        F.col("total_fare") / (F.col("total_duration_min") / 60)
    )
)

print(f"Aggregated training rows: {train_agg.count():,}")

# Aggregating test taxi trips by location, severity, day, precipitation type and time of day
test_agg = (
    test_df
    .groupBy(
        "PULocationID", "severity_cat", "week_day",
        "precip_type", "time_of_day"
    )
    .agg(
        F.sum("fare_amount").alias("total_fare"),
        F.sum("trip_duration_min").alias("total_duration_min"),
        F.count("*").alias("trip_count"),
        F.avg("severity_score").alias("avg_severity_score")
    )
    # Calculating earnings potential as fare earned per active hour
    .withColumn(
        "earnings_potential",
        F.col("total_fare") / (F.col("total_duration_min") / 60)
    )
)

print(f"Aggregated test rows: {test_agg.count():,}")

Aggregated training rows: 32,983


Aggregated test rows: 16,320


In [9]:
# Calculating total fare and trip duration across all training groups
overall_training_totals = train_agg.select(
    F.sum("total_fare").alias("total_fare"),
    F.sum("total_duration_min").alias("total_duration_min")
).collect()[0]

# Calculating the overall training set earnings per active hour
overall_training_earnings = (
    overall_training_totals["total_fare"] / (overall_training_totals["total_duration_min"] / 60)
)

print(f"Global training earnings baseline: ${overall_training_earnings:.2f}/hour")

# Setting thresholds for the trip count and duration adjustments
trips_thresh = 30       
duration_thresh = 60 

def add_adjusted_earnings(df):
    """
    This function shrinks the raw earnings_potential towards the overall training set
    earnings baseline based on the number of trips and total trip duration
    in each group to produce more stable estimates.
 
    Parameters:
    df: Current aggregated PySpark DataFrame

    Returns:
    PySpark DataFrame with the adj_earnings_potential feature added

    """

    return (
        df
        # Calculating the weight based on the number of trips in each group
        .withColumn("trip_reliability", 
                    F.col("trip_count") / 
                    (F.col("trip_count") + F.lit(trips_thresh))
                    )
        
        # Calculating the weight based on total trip duration in each group
        .withColumn("duration_reliability", 
                    F.col("total_duration_min") / 
                    (F.col("total_duration_min") + F.lit(duration_thresh))
                    )

        # Combining the trip count and duration weights
        .withColumn("reliability_weight", 
                    (F.col("trip_reliability") + F.col("duration_reliability")) 
                     / F.lit(2)
                     )

        # Shrinking group earnings potential towards the training set baseline
        # Reference: 
        # D. Martin, “Stacked turtles,” Stacked Turtles, Dec. 26, 2018. 
        # https://kiwidamien.github.io/shrinkage-and-empirical-bayes-to-improve-inference.html.
        .withColumn(
            "adj_earnings_potential",
            (F.col("reliability_weight") * F.col("earnings_potential"))
            + (
                (F.lit(1) - F.col("reliability_weight")) 
                * F.lit(overall_training_earnings)
                )
        )
    )

train_agg = add_adjusted_earnings(train_agg)
test_agg = add_adjusted_earnings(test_agg)

Global training earnings baseline: $68.15/hour


In [10]:
# Dropping unecessary columns

def drop_intermediates(df):

    """
    This function removes the columns used for feature engineering that
    are no longer required in the final datasets.

    Parameters: Aggregated PySpark DataFrame

    Returns: 
    PySpark DataFrame with the specified columns removed

    """

    return df.drop("trip_reliability", "duration_reliability", "reliability_weight")

train_agg = drop_intermediates(train_agg)
test_agg = drop_intermediates(test_agg)

In [11]:
# Saving the final training and test datasets as Parquet files

train_path = Path("../data/processed/train_agg.parquet")
test_path = Path("../data/processed/test_agg.parquet")

train_agg.write.mode("overwrite").parquet(str(train_path))
test_agg.write.mode("overwrite").parquet(str(test_path))

train_agg = spark.read.parquet(str(train_path))
test_agg = spark.read.parquet(str(test_path))

print(f"Aggregated traning rows: {train_agg.count():,}, columns: {len(train_agg.columns)}")
print(f"Aggregated test rows: {test_agg.count():,}, columns: {len(test_agg.columns)}")

Aggregated traning rows: 32,983, columns: 11
Aggregated test rows: 16,320, columns: 11
